# 眼科图像分类四组消融与框架对比实验
目的：分别评估两个框架中的AKConv改动，并观察框架实现差异。项目不是多智能体系统。

| 框架 | 原生模型 | AKConv变体 |
|---|---|---|
| YOLO11本地框架 | baseline | akconv |
| YOLO26本地框架 | yolo26n | yolo26n_akconv |

同框架原生与AKConv的差异是消融对比；跨框架结果不是纯网络结构对比。本地两版分类YAML骨干配置相同。

## 使用顺序
1. 选择Python (yolov10)内核，运行配置、数据检查、环境检查和训练函数定义。
2. 查看完成状态，只运行所需模型的训练单元。四个训练单元会启动正式50轮训练，不要无意点击全部运行。
3. 训练中保持内核运行；单元输出每5秒刷新，完整记录保留在log中。中断会停止本单元启动的子进程及加载进程。
4. 训练结束后运行汇总、曲线和混淆矩阵单元；重新打开Notebook不需要重新训练即可查看磁盘结果。
5. 每次重新运行配置会生成新目录；重复运行同一个模型不会覆盖结果。文件已在JupyterLab打开时，请从磁盘重新加载，不要用旧页面覆盖修改。

正式实验默认：全量5110训练图、1163验证图、50轮、224输入、batch32、seed0、8数据加载进程、FP32、随机初始化、AdamW。
旧AMP实验只作为历史参考；仅完成代码集成或小样本验证不代表完成正式消融。

## 统一配置

In [ ]:
from pathlib import Path
import sys, os, json, csv, subprocess, time
ROOT = Path(r'C:/Users/47449/Desktop/yolo/Intelligent_diagnosis_system')
PYTHON = Path(r'D:/Anaconda3/envs/yolo26/python.exe')
OLD_SOURCE = ROOT / 'ultralytics-main'
NEW_SOURCE = Path(r'C:/Users/47449/Desktop/yolo/yolo26/ultralytics-main')
DATA = ROOT / 'data'
CONFIG = ROOT / 'runs' / 'notebook_config'
CONFIG.mkdir(parents=True, exist_ok=True)
EPOCHS, IMGSZ, BATCH, SEED = 50, 224, 32, 0
# 四组统一FP32，避免AKConv半精度验证问题；不可直接与旧AMP耗时混比。
AMP = False
WORKERS = 8
CACHE = False
FRACTION = 1.0
RUN_ID = time.strftime('%Y%m%d_%H%M%S') + f'_w{WORKERS}_fp32_{time.time_ns()%1000000:06d}'
OUTPUT = ROOT / 'runs' / 'comparison_notebook' / RUN_ID
SPECS = {
 'baseline': (OLD_SOURCE, OLD_SOURCE/'ultralytics/cfg/models/11/yolo11-cls.yaml'),
 'akconv': (OLD_SOURCE, OLD_SOURCE/'ultralytics/cfg/models/11/yolo11-cls-AKConv.yaml'),
 'yolo26n': (NEW_SOURCE, NEW_SOURCE/'ultralytics/cfg/models/26/yolo26-cls.yaml'),
 'yolo26n_akconv': (NEW_SOURCE, NEW_SOURCE/'ultralytics/cfg/models/26/yolo26-cls-AKConv.yaml'),
}
env = dict(os.environ, YOLO_CONFIG_DIR=str(CONFIG), PYTHONIOENCODING='utf-8', PYTHONPATH=r'C:/Users/47449/Desktop/yolo/experiment_packages')
env["MPLBACKEND"] = "Agg"
print('内核:',sys.executable,'训练解释器:',PYTHON,'新实验输出:',OUTPUT,sep='\n')
assert 1 <= WORKERS <= 20, "WORKERS应在1到20之间，默认8"
print(f"加速版已加载：WORKERS={WORKERS}，AMP={AMP}，batch={BATCH}")

import runpy
BENCH = runpy.run_path(str(ROOT / 'scripts' / 'experiment_workbench.py'))
SETTINGS = dict(epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    amp=AMP, half=False, workers=WORKERS, cache=CACHE, fraction=FRACTION,
    pretrained=False, optimizer='AdamW', lr0=0.000833, lrf=0.01,
    momentum=0.9, weight_decay=0.0005, warmup_epochs=3.0,
    warmup_momentum=0.8, warmup_bias_lr=0.1, deterministic=True,
    cos_lr=False, dropout=0.0, auto_augment='randaugment', erasing=0.4, fliplr=0.5)
print('统一训练参数:', SETTINGS)
from IPython.display import display, Image


## 数据检查与内容指纹

In [ ]:
CLASSES = ['A','C','D','G','H','M','N','O']
SNAPSHOT = BENCH['dataset_snapshot'](DATA, CLASSES)
for split, counts in SNAPSHOT['counts'].items():
    print(split, counts, '总数:', sum(counts.values()))
print('数据内容SHA256:', SNAPSHOT['content_fingerprint_sha256'])
if EPOCHS != 50 or FRACTION != 1.0: print('注意：当前是验证配置，不是正式全量50轮实验')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'Microsoft YaHei','axes.unicode_minus':False})
fig, ax = plt.subplots(figsize=(10,4.5))
x = list(range(len(CLASSES)))
ax.bar([i-.2 for i in x], [SNAPSHOT['counts']['train'][c] for c in CLASSES], .4, label='训练集', color='#2563EB')
ax.bar([i+.2 for i in x], [SNAPSHOT['counts']['val'][c] for c in CLASSES], .4, label='验证集', color='#169B83')
ax.set_xticks(x, CLASSES); ax.set_ylabel('图像数'); ax.set_title('八类图像分布'); ax.legend(); ax.grid(axis='y',alpha=.2); ax.set_axisbelow(True)
fig.tight_layout(); (ROOT/'runs'/'report_figures').mkdir(exist_ok=True)
fig.savefig(ROOT/'runs'/'report_figures'/'dataset_distribution.png',dpi=240); plt.show()
print('正常类占比较高；实验报告应讨论类别不均衡。数据来源、患者级划分和标签转换需另行提供，不能从目录数量推断。')

## 四组环境与模型结构检查

In [ ]:
assert PYTHON.is_file(), '训练解释器不存在'
for tag,(source,yaml) in SPECS.items():
    assert yaml.is_file(), yaml
    probe = f"import sys; sys.path.insert(0,{str(source)!r}); import torch,ultralytics; from ultralytics import YOLO; m=YOLO({str(yaml)!r},task='classify'); print('SOURCE',ultralytics.__file__); print('CUDA',torch.cuda.is_available()); print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Unavailable'); print('AKConv layers',sum(type(x).__name__=='AKConv' for x in m.model.modules())); print('MODEL',m.task)"
    result=subprocess.run([str(PYTHON),'-c',probe],env=env,cwd=str(ROOT),capture_output=True,text=True,encoding='utf-8',errors='replace')
    print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f'{tag}环境检查失败，详见上方错误')
print('四组配置均可加载。模型配置的类别数可能不同；参数量以训练后8类权重为准。')

## 定义训练函数与查看已有进度

In [ ]:
def train_one(tag):
    source, model_yaml = SPECS[tag]
    # SETTINGS在配置单元固定；改参数后必须重跑配置单元。
    return BENCH['run_training'](tag, source, model_yaml, ROOT, OUTPUT, PYTHON, env, dict(SETTINGS), SNAPSHOT)
print('训练函数已定义；此单元不启动训练。')

In [ ]:
import pandas as pd
status = pd.DataFrame(BENCH['overview'](ROOT, SETTINGS))
if status.empty: print('还没有实验记录')
else: display(status)
print('完整同配置表示已记录参数一致；仍需核对数据来源、框架版本和最终验证状态。')

## 训练 baseline

In [ ]:
# 正式训练：运行此单元才会启动。
train_one('baseline')

## 训练 akconv

In [ ]:
# 正式训练：运行此单元才会启动。
train_one('akconv')

## 训练 yolo26n

In [ ]:
# 正式训练：运行此单元才会启动。
train_one('yolo26n')

## 训练 yolo26n_akconv

In [ ]:
# 正式训练：运行此单元才会启动。
train_one('yolo26n_akconv')

## 结果汇总与参数一致性检查

In [ ]:
import pandas as pd
runpy.run_path(str(ROOT / 'scripts' / 'export_comparison.py'), run_name='__main__', init_globals={'TARGET_SETTINGS': SETTINGS})
comparison = pd.read_csv(ROOT / 'runs' / 'comparison_summary.csv')
display(comparison[['model','best_top1_pct','best_top1_epoch','top5_at_best_top1_pct','amp','workers','matching_settings','completion_verified']])

In [ ]:
# 同框架的消融增益，单位为百分点；不同配置不计算增益。
pairs = [('YOLO11框架','YOLO11n-cls','YOLO11n-cls + AKConv'),
         ('YOLO26框架','YOLO26n-cls','YOLO26n-cls + AKConv')]
rows = []
for framework, original, modified in pairs:
    selected = comparison[comparison['model'].isin([original,modified])]
    if len(selected)!=2:
        rows.append({'框架':framework,'状态':'缺少完整实验','Top-1变化(pp)':None})
        continue
    selected = selected.set_index('model')
    a,b = selected.loc[original], selected.loc[modified]
    if not (bool(a['matching_settings']) and bool(b['matching_settings'])):
        rows.append({'框架':framework,'状态':'与当前统一配置不符，先补跑','Top-1变化(pp)':None})
        continue
    rows.append({'框架':framework,'状态':'参数一致；数据与最终验证需结合manifest核验',
                 'Top-1变化(pp)':round(b['best_top1_pct']-a['best_top1_pct'],3)})
display(pd.DataFrame(rows))


## 对比曲线

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT/'runs'/'comparison_curves.png'),width=1100))

## 真实验证混淆矩阵

In [ ]:
for _, row in comparison.iterrows():
    folder = (ROOT / row['source_csv']).parent
    print(row['model'], '|', folder)
    for name in ['confusion_matrix_normalized.png','confusion_matrix.png']:
        image = folder / name
        if image.exists():
            display(Image(filename=str(image),width=780))
            break
    else: print('缺少实际验证混淆矩阵；不能从准确率CSV推导。')
print('典型正确/错误样例可查看同目录val_batch*_labels.jpg与val_batch*_pred.jpg；不能把图中类别差异直接解释为医学根因。')

## 指标口径与报告检查清单
- Top-1为最高概率类别是否正确；Top-5为真实类别是否进入前五。八类任务中Top-5不能替代Top-1。
- 最佳Top-1、同轮Top-5、末轮Top-5及最高Top-5分别记录，不混用指标。
- CSV时间是训练累计时间；manifest中的wall_seconds还含最终验证及资源统计，二者都不是单图推理延迟。
- manifest记录解释器、PyTorch、框架版本与位置、GPU、YAML哈希、数据内容指纹、实际8类权重参数量与GFLOPs。缺失记录留空，不补造。
- 原生模型必须没有AKConv层，变体必须含AKConv；保存后的权重再次加载核验。
- 两个框架的数据增强及训练实现可能不同，即使参数一致，也不能声称全部差异来自骨干结构。
- 单种子只说明本次运行；若要结论更稳健，可另跑seed1、seed2，分别记录，不能混入同一条曲线。
- AKConv来自Lv等人 AKConv Convolutional Kernel with Arbitrary Sampled Shapes and Arbitrary Number of Parameters，属于引用集成工作。
- 课程报告仍需补充文献引用、数据来源、类别召回率和错误样例分析、系统功能测试、部署证据；本Notebook不自动宣称已完成全部课程要求。